In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))
from utils.spark_session import get_spark
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, StringType, DoubleType

spark =get_spark()
BRONZE_PATH = '../data/bronze'
SILVER_PATH = '../data/silver' 

df_bronze = spark.read.format('delta').load(BRONZE_PATH)

print("Null counts in key columns:")
df_bronze.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in ['is_violent_recid', 'v_decile_score', 'age', 'priors_count14', 'race']
]).show()

df_bronze.agg(
    F.min('v_decile_score').alias('min_score'),
    F.max('v_decile_score').alias('max_score'),
)


26/04/07 10:18:57 WARN Utils: Your hostname, Arjuns-MacBook-Air.local resolves to a loopback address: 127.0.0.1; using 10.0.0.234 instead (on interface en0)
26/04/07 10:18:57 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Ivy Default Cache set to: /Users/arjunpillai/.ivy2/cache
The jars for the packages stored in: /Users/arjunpillai/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-64772ae1-a894-415e-99d2-f58d625c0288;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.1.0 in central
	found io.delta#delta-storage;3.1.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 78ms :: artifacts dl 3ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.1.0 from central in [default]
	io.delta#delta-storage;3.1.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	-------------------------------------------------------------

:: loading settings :: url = jar:file:/Users/arjunpillai/Desktop/recidivai/.venv/lib/python3.9/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


26/04/07 10:18:58 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/07 10:18:58 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Null counts in key columns:


26/04/07 10:19:02 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+----------------+--------------+---+--------------+----+
|is_violent_recid|v_decile_score|age|priors_count14|race|
+----------------+--------------+---+--------------+----+
|               0|             0|  0|             0|   0|
+----------------+--------------+---+--------------+----+



DataFrame[min_score: int, max_score: int]

In [2]:
df_silver = (
    df_bronze
    .filter(F.col('is_violent_recid').isNotNull())
    .filter(F.col('v_decile_score').between(1, 10))
    .withColumn('age', F.col('age').cast(IntegerType()))
    .withColumn('priors_count14', F.col('priors_count14').cast(IntegerType()))
    .withColumn('juv_fel_count', F.col('juv_fel_count').cast(IntegerType()))
    .withColumn('juv_misd_count', F.col('juv_misd_count').cast(IntegerType()))
    .withColumn('is_violent_recid', F.col('is_violent_recid').cast(IntegerType()))
    .withColumn('v_decile_score', F.col('v_decile_score').cast(IntegerType()))
    .dropDuplicates(['id'])
    .select(
        'id',
        'is_violent_recid',
        'v_decile_score',
        'race',
        'age',
        'sex',
        'age_cat',
        'juv_fel_count',
        'juv_misd_count',
        'priors_count14',
        'c_charge_degree'
    )
)

df_silver.printSchema()
print(f'Silver row count: {df_silver.count()}')

root
 |-- id: integer (nullable = true)
 |-- is_violent_recid: integer (nullable = true)
 |-- v_decile_score: integer (nullable = true)
 |-- race: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- sex: string (nullable = true)
 |-- age_cat: string (nullable = true)
 |-- juv_fel_count: integer (nullable = true)
 |-- juv_misd_count: integer (nullable = true)
 |-- priors_count14: integer (nullable = true)
 |-- c_charge_degree: string (nullable = true)

Silver row count: 4743


In [3]:
bronze_count = df_bronze.count()
silver_count = df_silver.count()
dropped = bronze_count - silver_count

print('=== Silve Layer Data Quality Report ===')
print(f'Bronze rows: {bronze_count}')
print(f'Silver rows: {silver_count}')
print(f'Rows dropped: {dropped} ({(dropped/bronze_count)*100:.1f}%)')
print()

print('Remaining nulls in Silver (should all be 0):')
df_silver.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in ['is_violent_recid', 'age', 'priors_count14', 'race']
]).show()

print('Target Distribution in Silver:')
df_silver.groupBy('is_violent_recid').count().show()

=== Silve Layer Data Quality Report ===
Bronze rows: 4743
Silver rows: 4743
Rows dropped: 0 (0.0%)

Remaining nulls in Silver (should all be 0):
+----------------+---+--------------+----+
|is_violent_recid|age|priors_count14|race|
+----------------+---+--------------+----+
|               0|  0|             0|   0|
+----------------+---+--------------+----+

Target Distribution in Silver:
+----------------+-----+
|is_violent_recid|count|
+----------------+-----+
|               1|  819|
|               0| 3924|
+----------------+-----+



In [4]:
(
    df_silver
    .write
    .format('delta')
    .mode('overwrite')
    .save(SILVER_PATH)
)

print(f'Silver layer written to {SILVER_PATH}')

df_silver_check = spark.read.format('delta').load(SILVER_PATH)
df_silver_check.show(5)

Silver layer written to ../data/silver
+---+----------------+--------------+----------------+---+----+---------------+-------------+--------------+--------------+---------------+
| id|is_violent_recid|v_decile_score|            race|age| sex|        age_cat|juv_fel_count|juv_misd_count|priors_count14|c_charge_degree|
+---+----------------+--------------+----------------+---+----+---------------+-------------+--------------+--------------+---------------+
|  1|               0|             1|           Other| 69|Male|Greater than 45|            0|             0|             0|              F|
|  3|               1|             1|African-American| 34|Male|        25 - 45|            0|             0|             0|              F|
|  5|               0|             6|African-American| 23|Male|   Less than 25|            0|             1|             1|              F|
|  6|               0|             1|           Other| 43|Male|        25 - 45|            0|             0|             